In [2]:
"""
─────────────────
Uses Donut (Document Understanding Transformer) to extract all fillable
fields from a form image and output them as structured JSON.

NOTE: Donut is a seq2seq model — it reads the whole document and outputs
structured text. It does NOT produce pixel-level bounding boxes (that is
an open feature request on the original repo). What it gives you is:
  • field labels
  • field types  (checkbox / input line / text area)
  • current values (if any)

Install:
    pip install transformers torch sentencepiece Pillow
"""

import os
import re
import json
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_FOLDER = os.path.join(os.path.dirname("*"), "data")
IMAGE_NAME  = "form329.png"          # ← change to any form image
IMAGE_PATH  = os.path.join(DATA_FOLDER, IMAGE_NAME)

MODEL_ID    = "naver-clova-ix/donut-base-finetuned-docvqa"

# ─────────────────────────────────────────────
# LOAD MODEL
# ─────────────────────────────────────────────

def load_model():
    print("Loading Donut model (first run downloads ~900MB of weights)...")
    processor = DonutProcessor.from_pretrained(MODEL_ID)
    model     = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
    device    = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    print(f"Ready on {device}\n")
    return processor, model, device

# ─────────────────────────────────────────────
# SINGLE QUESTION → ANSWER
# ─────────────────────────────────────────────

def ask(image: Image.Image, question: str, processor, model, device) -> str:
    task_prompt       = f"<s_docvqa><s_question>{question}</s_question><s_answer>"
    decoder_input_ids = processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors="pt"
    ).input_ids
    pixel_values = processor(image.convert("RGB"), return_tensors="pt").pixel_values

    with torch.no_grad():
        outputs = model.generate(
            pixel_values.to(device),
            decoder_input_ids=decoder_input_ids.to(device),
            max_length=512,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    raw = processor.batch_decode(outputs.sequences)[0]
    raw = raw.replace(processor.tokenizer.eos_token, "")
    raw = raw.replace(processor.tokenizer.pad_token, "")
    raw = re.sub(r"<s_docvqa>.*?<s_answer>", "", raw, flags=re.DOTALL)
    raw = re.sub(r"</s_answer>.*", "", raw, flags=re.DOTALL)
    return raw.strip()

# ─────────────────────────────────────────────
# EXTRACT ALL FORM FIELDS
# ─────────────────────────────────────────────

FIELD_QUESTIONS = {
    "input_lines": (
        "List all the underline input fields in this form where a user "
        "would write text such as name, date, result, or signature."
    ),
    "checkboxes": (
        "List all the checkbox or yes/no options available in this form."
    ),
    "sections": (
        "What are the main sections or question labels in this form?"
    ),
}

def extract_fields(image: Image.Image, processor, model, device) -> dict:
    results = {}
    for field_type, question in FIELD_QUESTIONS.items():
        print(f"  Asking about: {field_type}...")
        answer = ask(image, question, processor, model, device)
        results[field_type] = answer
    return results

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    print(f"Loading image: {IMAGE_PATH}\n")
    image = Image.open(IMAGE_PATH)

    processor, model, device = load_model()

    print("Extracting form fields via Donut...\n")
    fields = extract_fields(image, processor, model, device)

    # ── Pretty print ────────────────────────────
    print("\n" + "="*60)
    print("  FORM FIELD EXTRACTION RESULTS")
    print("="*60)
    for field_type, answer in fields.items():
        print(f"\n[ {field_type.upper().replace('_', ' ')} ]")
        print(answer)

    # ── Save as JSON ─────────────────────────────
    output = {
        "image":  IMAGE_NAME,
        "model":  MODEL_ID,
        "fields": fields,
    }
    out_path = os.path.join(os.path.dirname("*"), "form_fields.json")
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n\nResults saved → {out_path}")


if __name__ == "__main__":
    main()

Loading image: data/form329.png

Loading Donut model (first run downloads ~900MB of weights)...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Ready on cpu

Extracting form fields via Donut...

  Asking about: input_lines...
  Asking about: checkboxes...
  Asking about: sections...

  FORM FIELD EXTRACTION RESULTS

[ INPUT LINES ]
version's mets level

[ CHECKBOXES ]
20

[ SECTIONS ]
section xiv - mets testing


Results saved → form_fields.json
